In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.window import Window

from config import DATASETS, DEDUP_KEYS, bronze_table, silver_table

def classify_rating(stars):
    if stars is None:
        return None
    if stars < 3:
        return "Baixa"
    if stars < 4:
        return "Media"
    return "Alta"


rating_band_udf = F.udf(classify_rating, StringType())

In [ ]:
for dataset in DATASETS:
    bronze_df = spark.table(bronze_table(dataset))

    window = Window.partitionBy(*DEDUP_KEYS[dataset]).orderBy(F.col("_ingested_at").desc())

    silver_df = (
        bronze_df
        .withColumn("_row_number", F.row_number().over(window))
        .filter(F.col("_row_number") == 1)
        .drop("_row_number", "_source_file", "_ingested_at")
    )

    if dataset == "business":
        silver_df = silver_df.withColumn("rating_band", rating_band_udf(F.col("stars")))

    silver_df.write.format("delta").mode("overwrite").saveAsTable(silver_table(dataset))

    print(f"{dataset}: {silver_df.count()} linhas gravadas em {silver_table(dataset)}")